In [2]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)

chat_model = ChatHuggingFace(llm=llm)

In [ ]:
response = chat_model.invoke("Tell me a joke about nutrition in one line.")


content='I’m on a “see‑food” diet—if it looks good, I’m sure it’s perfectly balanced!' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 100, 'prompt_tokens': 77, 'total_tokens': 177}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_fa36a6363905c2b568a1', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--019e0c52-23df-75c0-8cfc-77e78b08cc71-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 77, 'output_tokens': 100, 'total_tokens': 177}


In [4]:
print(response.content)

I’m on a “see‑food” diet—if it looks good, I’m sure it’s perfectly balanced!


In [8]:
from pydantic import BaseModel
from langchain_core.output_parsers import PydanticOutputParser
from langchain_core.prompts import ChatPromptTemplate

class Joke(BaseModel):
    joke: str
    explanation: str

# Create the parser
parser = PydanticOutputParser(pydantic_object=Joke)

# Create a proper prompt template with format instructions
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant that tells jokes. {format_instructions}"),
    ("user", "{query}")
])

# Build the chain
chain = prompt | chat_model | parser

# Invoke with proper input
response = chain.invoke({
    "query": "Tell me a joke about nutrition in one line with a little explanation.",
    "format_instructions": parser.get_format_instructions()
})

print(f"Joke: {response.joke}")
print(f"Explanation: {response.explanation}")

Joke: I told my salad it was going to be a 'leafy' day, but it just couldn't 'kale' my expectations!
Explanation: The joke plays on the word 'leafy' for greens and the pun 'kale' sounding like 'kill', suggesting the salad couldn't meet the high expectations.


In [ ]:
print(response)1

In [15]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.config import get_stream_writer
from sympy import refine

from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from dotenv import load_dotenv
load_dotenv()  # Load environment variables from .env file

llm = HuggingFaceEndpoint(
    repo_id="openai/gpt-oss-120b",
    task="text-generation",
    max_new_tokens=512,
    do_sample=False,
    repetition_penalty=1.03,
    provider="auto",  # let Hugging Face choose the best provider for you
)

chat_model = ChatHuggingFace(llm=llm)

class joke(BaseModel):
    joke: str
    reason: str

class State(TypedDict):
    topic: str
    joke: joke
    refine_joke: str


def generate_joke(state: State):
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a helpful assistant that tells jokes. {format_instructions}"),
        ("user", "Generate a joke about {topic} and explain why it's funny.")
    ])

    writer = get_stream_writer()
    writer({"status": "thinking of a joke..."})
    chain = prompt | chat_model | PydanticOutputParser(pydantic_object=joke)
    response = chain.invoke({
        "topic": state['topic'],
        "format_instructions": PydanticOutputParser(pydantic_object=joke).get_format_instructions()
    })
    return {"joke": response}

def refine_joke(state: State):
    prompt = f"Refine the following joke to make it funnier: {state['joke'].joke}"
    writter = get_stream_writer()
    writter({"status": "refining the joke..."})
    response = chat_model.invoke(prompt)
    return {"refine_joke": response.content}


graph = (
    StateGraph(State)
    .add_node(generate_joke)
    .add_node(refine_joke)
    .add_edge(START, "generate_joke")
    .add_edge("generate_joke", "refine_joke")
    .add_edge("refine_joke", END)
    .compile()
)

for chunk in graph.stream(
    {"topic": "ice cream"},
    stream_mode=["updates", "custom", "messages"],
    version="v2",
):
    if chunk["type"] == "updates":
        for node_name, state in chunk["data"].items():
            if node_name in ["generate_joke"]:
                print(f"Node {node_name} updated: {state}")

            print("\n----------\n")
            if node_name in ["refine_joke"]:
                print(f"Node {node_name} updated: {state}")
    if chunk["type"] == "custom":
        # UpdatesStreamPart — only the changed keys from each node
        print(f"Status: {chunk['data']['status']}")

    elif chunk['type'] == "messages":
        # MessagesStreamPart — (message_chunk, metadata) from LLM calls
        msg, metadata = chunk["data"]
        print(msg.content, end="", flush=True)



Status: thinking of a joke...
{
  "joke": "Why did the ice cream go to school? Because it wanted to be a little smarter and avoid a brain freeze!",
  "reason": "The joke works by personifying ice cream—giving it human goals like going to school—while playing on the double meaning of \"brain freeze.\" The punchline mixes the literal desire to be smarter with the literal consequence of eating ice cream too fast (a brain freeze), creating a humorous pun."
}Node generate_joke updated: {'joke': joke(joke='Why did the ice cream go to school? Because it wanted to be a little smarter and avoid a brain freeze!', reason='The joke works by personifying ice cream—giving it human goals like going to school—while playing on the double meaning of "brain freeze." The punchline mixes the literal desire to be smarter with the literal consequence of eating ice cream too fast (a brain freeze), creating a humorous pun.')}

----------

Status: refining the joke...
**Why did the ice‑cream go to school?**  
B